# S19 · Cinemática de un brazo 2R (DOBOT Magician)

**Objetivo (RA4-a/c/d):** resolver la **cinemática directa e inversa** de un brazo plano 2R, dibujar su **espacio de trabajo** y entender sus problemas (dos soluciones, singularidades, fuera de alcance). Cierra con una **actividad**: probar el código y añadir una conclusión.


### Python en un vistazo (lo justo para esta sesión)

| Lo que verás | Qué significa |
|---|---|
| `# texto` | **Comentario**: Python lo ignora; es una nota para ti |
| `nombre = valor` | **Guarda** un valor con un nombre |
| `import numpy as np` | Carga **NumPy** (números y matrices) |
| `np.cos(x)` / `np.sin(x)` | **Coseno** / **seno** de un ángulo (en radianes) |
| `np.radians(grados)` | Convierte **grados → radianes** |
| `def nombre(...):` | Crea una **función** |
| `for ... in ...:` | **Repetición** (bucle) |
| `print(...)` | Escribe en pantalla |

Ejecuta las celdas de arriba a abajo; los comentarios `#` explican cada línea.


## 1. Cinemática directa (FK): de los ángulos a la posición

Un **brazo plano 2R** tiene dos eslabones y dos articulaciones:

- `l1 = 150 mm` (hombro → codo) y `l2 = 120 mm` (codo → muñeca).
- Dos ángulos: `θ1` (gira el primer eslabón) y `θ2` (gira el segundo **respecto al primero**).

La **cinemática directa** responde a: *«sé los ángulos → ¿dónde está la punta?»*

$$x = l_1\cos\theta_1 + l_2\cos(\theta_1+\theta_2) \qquad y = l_1\sin\theta_1 + l_2\sin(\theta_1+\theta_2)$$


In [ ]:
%matplotlib inline
import numpy as np                # NumPy: cos, sin, pi, arccos, etc.
import matplotlib.pyplot as plt   # matplotlib: para dibujar el brazo y la trayectoria

L1, L2 = 150.0, 120.0             # longitudes de los dos eslabones, en mm

def fk_2r(theta1, theta2):
    """Punta (x, y) del brazo 2R a partir de los ángulos en radianes."""
    x = L1*np.cos(theta1) + L2*np.cos(theta1 + theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1 + theta2)
    return x, y

# Ejemplo «a mano»: brazo estirado hacia delante (theta1=0°, theta2=0°)
x, y = fk_2r(np.radians(0), np.radians(0))
print("Estirado hacia delante -> punta en", (round(x, 1), round(y, 1)), "mm")   # (270.0, 0.0)

# Brazo estirado hacia arriba (theta1=90°, theta2=0°)
x, y = fk_2r(np.radians(90), np.radians(0))
print("Estirado hacia arriba   -> punta en", (round(x, 1), round(y, 1)), "mm")   # (0.0, 270.0)


**Idea clave:** con el brazo **estirado** (`θ2 = 0°`), la punta está siempre a `l1 + l2 = 270 mm` de la base, sea cual sea `θ1`. Por eso **270 mm es el alcance máximo**. El DOBOT llega a **320 mm**, así que este brazo de ejemplo cabe dentro de su zona de trabajo.

Dibujemos el brazo en una postura cualquiera para verlo:


In [ ]:
def dibujar_brazo(theta1, theta2):
    """Dibuja el brazo 2R y marca la punta."""
    codo = (L1*np.cos(theta1), L1*np.sin(theta1))
    punta = fk_2r(theta1, theta2)
    plt.figure(figsize=(5, 5))
    plt.plot([0, codo[0], punta[0]], [0, codo[1], punta[1]], "o-", lw=3)
    plt.plot(0, 0, "ks", ms=8, label="Base")
    plt.text(punta[0], punta[1], "  punta")
    plt.xlim(-300, 300); plt.ylim(-300, 300)
    plt.gca().set_aspect("equal"); plt.grid(True)
    plt.title(f"Brazo 2R  (θ1={np.degrees(theta1):.0f}°, θ2={np.degrees(theta2):.0f}°)")
    plt.show()

dibujar_brazo(np.radians(60), np.radians(45))


## 2. Cinemática inversa (IK): de la posición a los ángulos

La **cinemática inversa** hace el camino al revés: *«quiero la punta en (x, y) → ¿qué ángulos pongo?»*. Es el problema **difícil**, porque un mismo punto se alcanza con **dos posturas** («codo abajo» o «codo arriba»).


In [ ]:
def ik_2r(x, y):
    """Los dos pares de ángulos (codo abajo, codo arriba) para alcanzar (x, y)."""
    c2 = (x*x + y*y - L1*L1 - L2*L2) / (2*L1*L2)   # cos(theta2)
    c2 = np.clip(c2, -1, 1)                         # por errores de redondeo
    theta2_abajo  =  np.arccos(c2)                  # codo «abajo»
    theta2_arriba = -np.arccos(c2)                  # codo «arriba»
    def theta1(t2):
        return np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1 + L2*np.cos(t2))
    return (theta1(theta2_abajo), theta2_abajo), (theta1(theta2_arriba), theta2_arriba)

sol_abajo, sol_arriba = ik_2r(200, 100)   # queremos la punta en (200, 100) mm

print("Solución 1 (codo abajo):", tuple(round(np.degrees(a), 1) for a in sol_abajo))
print("Solución 2 (codo arriba):", tuple(round(np.degrees(a), 1) for a in sol_arriba))

# Comprobamos: las dos devuelven el mismo punto (200, 100)
print("FK(sol 1) =", tuple(round(v, 1) for v in fk_2r(*sol_abajo)))
print("FK(sol 2) =", tuple(round(v, 1) for v in fk_2r(*sol_arriba)))


## 3. Espacio de trabajo

El **espacio de trabajo** es el conjunto de puntos que la punta puede alcanzar. Para este brazo es un **anillo** entre `|l1 − l2| = 30 mm` y `l1 + l2 = 270 mm`. La circunferencia roja marca el alcance del DOBOT (320 mm).


In [ ]:
xs, ys = [], []
for t1 in np.linspace(-np.pi, np.pi, 90):   # barremos todos los ángulos posibles
    for t2 in np.linspace(-np.pi, np.pi, 90):
        x, y = fk_2r(t1, t2)
        xs.append(x); ys.append(y)

plt.figure(figsize=(5, 5))
plt.scatter(xs, ys, s=1, alpha=0.3)
plt.gca().add_patch(plt.Circle((0, 0), 320, fill=False, ls="--", color="red", label="alcance DOBOT 320 mm"))
plt.plot(0, 0, "ks", ms=6, label="Base")
plt.gca().set_aspect("equal"); plt.grid(True); plt.legend()
plt.xlabel("X (mm)"); plt.ylabel("Y (mm)")
plt.title("Espacio de trabajo del brazo 2R")
plt.show()


## 4. Singularidad y fuera de alcance

- **Singularidad:** con el brazo **totalmente estirado**, el robot «pierde» un grado de libertad (no puede moverse perpendicular a sí mismo sin un esfuerzo brusco).
- **Fuera de alcance:** un punto a más de 270 mm (320 en el DOBOT) **no se alcanza**.


In [ ]:
def alcanzable(x, y):
    r = np.sqrt(x*x + y*y)
    return abs(L1 - L2) <= r <= (L1 + L2)   # dentro del anillo de trabajo

print("Punto (200, 100) -> alcanzable:", alcanzable(200, 100))   # r ≈ 223.6 mm → sí
print("Punto (300, 300) -> alcanzable:", alcanzable(300, 300))   # r ≈ 424 mm → NO

# Singularidad: pide por IK un punto casi en el límite (l1 + l2 = 270 mm)
sol_abajo, _ = ik_2r(269.9, 0)
print("Para (269.9, 0), theta2 ≈", round(np.degrees(sol_abajo[1]), 2), "°  (casi 0 → brazo estirado)")


## 5. Actuadores, sensores y efectores: el caso de la ventosa

Una **ventosa** no «agarra»: **aspira**. La fuerza de sujeción es `F = P · A` (presión × área).

La del DOBOT: diámetro **Ø 20 mm** y **−35 kPa**. Calculamos la fuerza teórica:


In [ ]:
P = 35000.0                  # 35 kPa en pascales (Pa)
diametro = 20.0 / 1000.0     # 20 mm -> 0.020 m
radio = diametro / 2
area = np.pi * radio**2      # área del círculo, en m²
F = P * area                 # F = P · A  (newtons)
print("Fuerza teórica de la ventosa:", round(F, 1), "N")
print("Suficiente para un cubo ligero; no para una pieza pesada o porosa.")


## Actividad · Prueba el código y concluye

1. **Prueba el código** cambiando parámetros y volviendo a ejecutar:
   - **FK:** otros ángulos (p. ej. `θ1 = 45°`, `θ2 = 30°`) y mira dónde queda la punta.
   - **IK:** otro punto objetivo (p. ej. `(250, 0)` o `(100, 200)`) y sus dos soluciones.
   - **`alcanzable`:** un punto dentro y otro fuera del alcance.
2. **Añade una conclusión de 3–4 líneas**: qué problemas de cinemática has encontrado (dos soluciones, singularidad, fuera de alcance) y cómo se resuelven (CE 4a + 4b).
